<a href="https://colab.research.google.com/github/begumsoeba786-art/task1-ml-classification1/blob/main/Task3_Model_Deploymentd0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install pyngrok flask

In [ ]:
import pickle
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# Load and train
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train_scaled, y_train)

# Save model and scaler
with open('model_task1.pkl', 'wb') as f:
    pickle.dump(model, f)

with open('scaler_task1.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("Model saved successfully!")

Model saved successfully!


In [ ]:
# Write the Flask app to a file
flask_code = '''
from flask import Flask, request, jsonify
import pickle
import numpy as np

app = Flask(__name__)

# Load model and scaler
with open("model_task1.pkl", "rb") as f:
    model = pickle.load(f)

with open("scaler_task1.pkl", "rb") as f:
    scaler = pickle.load(f)

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message": "InternSpark AI Internship - Task 3",
        "author": "Soeba Begum",
        "model": "Breast Cancer Classifier",
        "endpoint": "/predict",
        "method": "POST"
    })

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        features = np.array(data["features"]).reshape(1, -1)
        scaled = scaler.transform(features)
        prediction = model.predict(scaled)[0]
        probability = model.predict_proba(scaled)[0]

        result = "Benign" if prediction == 1 else "Malignant"

        return jsonify({
            "prediction": int(prediction),
            "result": result,
            "confidence": round(float(max(probability)) * 100, 2),
            "status": "success"
        })
    except Exception as e:
        return jsonify({"error": str(e), "status": "failed"})

if __name__ == "__main__":
    app.run()
'''

with open('app.py', 'w') as f:
    f.write(flask_code)

print("Flask app created!")

Flask app created!


In [ ]:
dockerfile = '''FROM python:3.10-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install -r requirements.txt

COPY . .

EXPOSE 5000

CMD ["python", "app.py"]
'''

requirements = '''flask==2.3.2
scikit-learn==1.3.0
numpy==1.24.3
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

with open('requirements.txt', 'w') as f:
    f.write(requirements)

print("Dockerfile and requirements.txt created!")

Dockerfile and requirements.txt created!


In [ ]:
from pyngrok import ngrok
import subprocess, time

# Your real ngrok token
ngrok.set_auth_token("3E7u7mdHgooyRralugotNj3XQ45_4uUWCB2mSh92Hab7UKUUg")

# Start Flask
subprocess.Popen(["python", "app.py"])
time.sleep(5)

# Create public URL
public_url = ngrok.connect(5000)
print("="*50)
print("YOUR API IS LIVE AT:")
print(public_url)
print("="*50)

YOUR API IS LIVE AT:
NgrokTunnel: "https://undergrad-overture-gallows.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import requests

api_url = "https://undergrad-overture-gallows.ngrok-free.dev/predict"

test_data = {
    "features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776,
                 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053,
                 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587,
                 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0,
                 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]
}

# Add ngrok header to bypass warning page
headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true"
}

response = requests.post(api_url, json=test_data, headers=headers)

print("Status Code:", response.status_code)
print("Raw Response:", response.text)

if response.status_code == 200:
    result = response.json()
    print("="*40)
    print("API TEST RESULT")
    print("="*40)
    print(f"Prediction : {result['result']}")
    print(f"Confidence : {result['confidence']}%")
    print(f"Status     : {result['status']}")
else:
    print("Error - check Flask is still running!")

Status Code: 502
Raw Response: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymo

In [ ]:
import os
os.system("pkill -f app.py")
print("Cleared!")

Cleared!


In [ ]:
import subprocess, time

subprocess.Popen(["python", "app.py"])
time.sleep(5)
print("Flask started!")

Flask started!


In [ ]:
from pyngrok import ngrok

ngrok.kill()
ngrok.set_auth_token("3E7u7mdHgooyRralugotNj3XQ45_4uUWCB2mSh92Hab7UKUUg")

public_url = ngrok.connect(5000)
print("="*50)
print("NEW API URL:")
print(public_url)
print("="*50)

NEW API URL:
NgrokTunnel: "https://undergrad-overture-gallows.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import requests

api_url = "https://undergrad-overture-gallows.ngrok-free.dev/predict"

test_data = {
    "features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776,
                 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053,
                 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587,
                 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0,
                 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]
}

# Add ngrok header to bypass warning page
headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true"
}

response = requests.post(api_url, json=test_data, headers=headers)

print("Status Code:", response.status_code)
print("Raw Response:", response.text)

if response.status_code == 200:
    result = response.json()
    print("="*40)
    print("API TEST RESULT")
    print("="*40)
    print(f"Prediction : {result['result']}")
    print(f"Confidence : {result['confidence']}%")
    print(f"Status     : {result['status']}")
else:
    print("Error - check Flask is still running!")

Status Code: 502
Raw Response: <!DOCTYPE html>
<html class="h-full" lang="en-US" dir="ltr">
  <head>
    <meta charset="utf-8">
    <meta name="viewport" content="width=device-width, initial-scale=1">
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Regular-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-RegularItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-Medium-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/euclid-square/EuclidSquare-MediumItalic-WebS.woff" as="font" type="font/woff" crossorigin="anonymous" />
    <link rel="preload" href="https://assets.ngrok.com/fonts/ibm-plex-mono/IBMPlexMono-Text.woff" as="font" type="font/woff" crossorigin="anonymo

In [ ]:
import subprocess
import time
import os
from pyngrok import ngrok
import threading

# Kill anything running on port 5000
os.system("fuser -k 5000/tcp")
os.system("pkill -f app.py")
time.sleep(2)

# Check app.py exists
import os.path
if os.path.exists("app.py"):
    print("app.py found!")
else:
    print("app.py NOT found - creating it now...")
    flask_code = '''
from flask import Flask, request, jsonify
import pickle
import numpy as np

app = Flask(__name__)

with open("model_task1.pkl", "rb") as f:
    model = pickle.load(f)

with open("scaler_task1.pkl", "rb") as f:
    scaler = pickle.load(f)

@app.route("/", methods=["GET"])
def home():
    return jsonify({"message": "Task 3 API is running!", "status": "ok"})

@app.route("/predict", methods=["POST"])
def predict():
    try:
        data = request.get_json()
        features = np.array(data["features"]).reshape(1, -1)
        scaled = scaler.transform(features)
        prediction = model.predict(scaled)[0]
        probability = model.predict_proba(scaled)[0]
        result = "Benign" if prediction == 1 else "Malignant"
        return jsonify({
            "prediction": int(prediction),
            "result": result,
            "confidence": round(float(max(probability)) * 100, 2),
            "status": "success"
        })
    except Exception as e:
        return jsonify({"error": str(e), "status": "failed"})

if __name__ == "__main__":
    app.run(host="0.0.0.0", port=5000)
'''
    with open("app.py", "w") as f:
        f.write(flask_code)
    print("app.py created!")

# Check model files exist
if not os.path.exists("model_task1.pkl"):
    print("Model not found - retraining...")
    from sklearn.datasets import load_breast_cancer
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import train_test_split
    import pickle

    data = load_breast_cancer()
    X, y = data.data, data.target
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train_scaled, y_train)

    with open("model_task1.pkl", "wb") as f:
        pickle.dump(model, f)
    with open("scaler_task1.pkl", "wb") as f:
        pickle.dump(scaler, f)
    print("Model retrained and saved!")
else:
    print("Model files found!")

# Start Flask
process = subprocess.Popen(
    ["python", "app.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)
time.sleep(4)

# Check if Flask started
if process.poll() is None:
    print("Flask is running!")
else:
    out, err = process.communicate()
    print("Flask error:", err.decode())

# Start ngrok
ngrok.kill()
ngrok.set_auth_token("3E7u7mdHgooyRralugotNj3XQ45_4uUWCB2mSh92Hab7UKUUg")
public_url = ngrok.connect(5000)
print("="*50)
print("API IS LIVE AT:")
print(public_url)
print("="*50)

app.py NOT found - creating it now...
app.py created!
Model not found - retraining...
Model retrained and saved!
Flask is running!
API IS LIVE AT:
NgrokTunnel: "https://undergrad-overture-gallows.ngrok-free.dev" -> "http://localhost:5000"


In [ ]:
import requests

api_url = "https://undergrad-overture-gallows.ngrok-free.dev/predict"

test_data = {
    "features": [17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776,
                 0.3001, 0.1471, 0.2419, 0.07871, 1.095, 0.9053,
                 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587,
                 0.03003, 0.006193, 25.38, 17.33, 184.6, 2019.0,
                 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189]
}

headers = {
    "Content-Type": "application/json",
    "ngrok-skip-browser-warning": "true"
}

response = requests.post(api_url, json=test_data, headers=headers)

print("Status Code:", response.status_code)

if response.status_code == 200:
    result = response.json()
    print("="*40)
    print("API TEST RESULT")
    print("="*40)
    print(f"Prediction : {result['result']}")
    print(f"Confidence : {result['confidence']}%")
    print(f"Status     : {result['status']}")
else:
    print("Response:", response.text[:200])

Status Code: 200
API TEST RESULT
Prediction : Malignant
Confidence : 100.0%
Status     : success
